# 00 — Démarrer : comprendre les données avant de modéliser

**Question :** peut-on anticiper le nombre d'actes criminels déclarés par PDQ et par semaine ?
Ce notebook n'entraîne **aucun modèle** : il vérifie ce que mesure la cible, ce qui manque,
et découpe les données dans le temps.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent                  # le notebook est dans notebooks/ → la racine est le dossier parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)                 # cfg est maintenant un dictionnaire Python

SEED = cfg["experiment"]["seed"]
DATA_PATH = ROOT / cfg["data"]["path"]

raw = pd.read_csv(DATA_PATH, dtype={"PDQ": "string"})

print("Config :", cfg)
print("Lignes :", len(raw))
print("Première date :", raw["DATE"].min(), "| Dernière date :", raw["DATE"].max())
print()
print(raw.isna().sum())
raw.head()

## Nettoyage
On retire seulement les lignes sans DATE ou sans PDQ (impossibles à compter).
Les coordonnées GPS manquantes ne sont pas un problème : on compte par PDQ, pas par position.

In [ ]:
df = raw.copy()                                             # on garde raw intact (brut = non modifié)
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")    # texte → vraie date

print("Dates illisibles :", df["DATE"].isna().sum())
print("Lignes sans PDQ :", df["PDQ"].isna().sum())

df = df.dropna(subset=["DATE", "PDQ"])                      # on retire SEULEMENT ce qui empêche le comptage
print("Lignes gardées :", len(df), "sur", len(raw))

Prédire la semaine prochaine pour un poste fermé n’a aucun sens. On garde donc ceux qui ont au moins 100 actes depuis 2025

In [ ]:
P = cfg["panel"]
recents = df.loc[df["DATE"] >= P["active_since"], "PDQ"].value_counts()   # nb d'actes par PDQ depuis 2025
pdqs = sorted(recents[recents >= P["min_acts"]].index, key=int)          # trié 1, 3, 4… (pas "1", "10", "12"…)

print(len(pdqs), "PDQ actifs :", pdqs)
print("Exclus :", sorted(set(df["PDQ"]) - set(pdqs), key=int))

In [ ]:
par_jour = df.groupby(df["DATE"].dt.date).size()
par_jour.tail(21)

## Latence de déclaration
Les derniers jours du fichier sont incomplets (délai de saisie), pas une baisse de la criminalité.
On ignore les 14 derniers jours et on ne garde que des semaines entières.

In [ ]:
cutoff = df["DATE"].max() - pd.Timedelta(days=P["lag_days"])   # 2026-11-04 − 14 jours
semaine_du_cutoff = cutoff.to_period("W-SUN").start_time        # le lundi de la semaine qui contient le cutoff

# Si le cutoff tombe un dimanche, sa semaine est complète ; sinon on recule d'une semaine.
if cutoff.day_of_week == 6:
    derniere_semaine = semaine_du_cutoff
else:
    derniere_semaine = semaine_du_cutoff - pd.Timedelta(days=7)

print("Dernière date du fichier :", df["DATE"].max().date())
print("Cutoff (−14 jours)       :", cutoff.date(), cutoff.day_name())
print("Dernière semaine gardée  : du", derniere_semaine.date(), "au", (derniere_semaine + pd.Timedelta(days=6)).date())

## Panel PDQ × semaine
Une ligne = un PDQ actif pendant une semaine (lundi → dimanche), y = nombre d'actes déclarés.
Une semaine sans acte n'apparaît pas dans le CSV : on l'ajoute avec y = 0 (zéro explicite).

In [ ]:
df["SEMAINE"] = df["DATE"].dt.to_period("W-SUN").dt.start_time   # le lundi de la semaine de l'acte

print("Premier jour du fichier :", df["DATE"].min().date(), df["DATE"].min().day_name())
print("Première semaine        :", df["SEMAINE"].min().date())
df[["DATE", "SEMAINE"]].head()




In [ ]:
premiere_semaine = df["SEMAINE"].min()
if df["DATE"].min().day_of_week != 0:                  # le fichier ne commence pas un lundi…
    premiere_semaine = premiere_semaine + pd.Timedelta(days=7)   # …on saute cette semaine partielle

print("Panel : du", premiere_semaine.date(), "au", derniere_semaine.date())

In [ ]:
garde = df["PDQ"].isin(pdqs) & df["SEMAINE"].between(premiere_semaine, derniere_semaine)
comptes = df[garde].groupby(["SEMAINE", "PDQ"]).size()

print("Actes gardés :", garde.sum())
print("Combinaisons (semaine, PDQ) qui ont AU MOINS un acte :", len(comptes))

In [ ]:
semaines = pd.date_range(premiere_semaine, derniere_semaine, freq="7D")       # TOUS les lundis
grille = pd.MultiIndex.from_product([semaines, pdqs], names=["SEMAINE", "PDQ"])  # chaque semaine × chaque PDQ

panel = comptes.reindex(grille, fill_value=0).rename("y").reset_index()

print("Semaines :", len(semaines), "| PDQ :", len(pdqs), "| Lignes attendues :", len(semaines) * len(pdqs))
print("Lignes du panel :", len(panel))
print("Zéros ajoutés   :", (panel["y"] == 0).sum())
print("Total des actes :", panel["y"].sum(), "(doit égaler les actes gardés)")
panel.head()

In [ ]:
print(panel.loc[panel["y"] == 0, "PDQ"].value_counts())
panel["y"].describe()

## Découpage temporel
Train : semaines avant 2023 · Validation : 2023 · Test : 2024 et après (fermé jusqu'à la fin).
On n'explore QUE le train : regarder la validation ou le test pour trouver des idées les rendrait moins « frais ».

In [ ]:
S = cfg["split"]
train_end = pd.Timestamp(S["train_end"])
validation_end = pd.Timestamp(S["validation_end"])

masks = {
    "train":      panel["SEMAINE"] < train_end,
    "validation": (panel["SEMAINE"] >= train_end) & (panel["SEMAINE"] < validation_end),
    "test":       panel["SEMAINE"] >= validation_end,
}

for nom, m in masks.items():
    sous = panel[m]
    print(f"{nom:<11} {len(sous):>6} lignes | {sous['SEMAINE'].nunique():>3} semaines | "
          f"du {sous['SEMAINE'].min().date()} au {sous['SEMAINE'].max().date()}")

# Contrôle : chaque ligne est dans UN SEUL ensemble
total = sum(m.sum() for m in masks.values())
print("Total :", total, "=", len(panel), "?", total == len(panel))

In [ ]:
for nom in ["train", "validation"]:
    y = panel.loc[masks[nom], "y"]
    print(f"{nom:<11} moyenne {y.mean():6.2f} | médiane {y.median():5.1f}")

### Limite connue : fermeture du PDQ 22 (fin 2024)
Son territoire a été redistribué, surtout au PDQ 21 (+23 % d'actes sur 12 mois alors que la plupart des PDQ baissent).
Ce n'est pas une hausse de la criminalité mais un changement de ce que mesure « PDQ 21 ».
Cette rupture tombe dans la période de test : le modèle ne l'a jamais vue. À vérifier : erreur par PDQ.

## Analyse exploratoire (train seulement : 2015–2022)
Trois questions : où, quand dans l'année, et comment le volume évolue-t-il d'une année à l'autre ?

In [ ]:
import matplotlib.pyplot as plt

train = panel[masks["train"]].copy()       # on n'explore QUE le train
print("Lignes du train :", len(train))

In [ ]:
par_pdq = train.groupby("PDQ")["y"].mean().sort_values()

ax = par_pdq.plot(kind="barh", figsize=(7, 8))
ax.set_xlabel("Actes déclarés par semaine (moyenne)")
ax.set_ylabel("PDQ")
ax.set_title("Volume moyen par PDQ — train 2015–2022")
plt.tight_layout()
plt.show()

print("Les 5 plus chargés :", par_pdq.tail(5).round(1).to_dict())
print("Les 4 moins chargés :", par_pdq.head(4).round(1).to_dict())

**Constat — Où.** Le volume varie d'un facteur ~9 entre PDQ (38 : ~41/semaine ; 50 : ~5/semaine).
Le PDQ est donc l'information la plus importante pour le modèle (rôle de l'heure chez les vélos).
Une même erreur de 5 actes est faible pour le PDQ 38, mais énorme pour le PDQ 50 : il faudra regarder l'erreur par PDQ.

In [ ]:
par_mois = train.groupby(train["SEMAINE"].dt.month)["y"].mean()

ax = par_mois.plot(marker="o", figsize=(7, 4))
ax.set_xticks(range(1, 13))
ax.set_xlabel("Mois")
ax.set_ylabel("Actes par PDQ par semaine (moyenne)")
ax.set_title("Saisonnalité — train 2015–2022")
plt.tight_layout()
plt.show()

print(par_mois.round(1).to_dict())

**Constat — Quand.** Saison marquée : creux en janvier–février (~16), pic en septembre–octobre (~21), soit ~30 % d'écart.
Le mois est une information utile pour le modèle.

In [ ]:
par_annee = train.groupby(train["SEMAINE"].dt.year)["y"].mean()

ax = par_annee.plot(marker="o", figsize=(7, 4))
ax.set_xlabel("Année")
ax.set_ylabel("Actes par PDQ par semaine (moyenne)")
ax.set_title("Tendance annuelle — train 2015–2022")
plt.tight_layout()
plt.show()

print(par_annee.round(2).to_dict())

**Constat — Tendance.** Baisse de 2015 à 2020 (creux, année de pandémie), puis remontée nette en 2021–2022.
La tendance n'est pas une ligne droite : un modèle qui prolonge une tendance linéaire apprise sur 2015–2022
prédirait une baisse, alors que 2023 continue de monter. C'est pourquoi la validation (22,3) est plus haute que la moyenne du train (18,6).
Conséquence pour le notebook 01 : comparer à une référence simple, et se méfier d'une tendance « en ligne droite ».